<div class="alert alert-block alert-info">
<b>Welcome to SynEdu!</b> This talktorial is part of <b>SynEdu</b>, a lightweight teaching series built around the <b>Syn</b> ecosystem and <b>RDKit</b> for practical, reproducible cheminformatics.
</div>

<div class="alert alert-block alert-warning">
<b>Reproducibility first</b>: Keep runtime short, prefer small datasets, and pin dependencies (e.g., via <code>env/environment.yml</code>). Save version info alongside exported figures.
</div>

<div class="alert alert-block alert-success">
<b>By the end of this notebook</b>, you will produce clear, reviewer-friendly visualizations: mapped molecules with labels and a tiny reaction network (CRN-like graph).
</div>

# S08 · Visualization: mapped molecules + tiny CRN


## Authors and contributions

- Tieu-Long Phan, Peter Stadler group, Professur für Bioinformatik, Institut für Informatik, Universität Leipzig
- (Add contributors here)


<div class="alert alert-block alert-info">
<b>Cross-referencing</b>: When referring to another SynEdu notebook, use <b>Talktorial SXX</b> (e.g., <b>Talktorial S03</b>).
</div>


## Roadmap
- Concepts: why visualization matters
- Hands-on: draw mapped molecules and export a tiny CRN DOT


# Theory

Visualization is not just aesthetics:
- It makes mapping QC and rule extraction inspectable.
- It helps debug unexpected rule applications.

We use:
- RDKit depictions for molecules
- NetworkX + DOT for network structure


# Practical


In [ ]:
from __future__ import annotations

from pathlib import Path
import pandas as pd
import networkx as nx

from rdkit import Chem
from rdkit.Chem import Draw

# Optional: Syn ecosystem (kept optional for Paper 1)
try:
    import synkit  # type: ignore
    HAS_SYNKit = True
except Exception:
    HAS_SYNKit = False

OUT = Path("talktorials/out")
OUT.mkdir(parents=True, exist_ok=True)

import rdkit
import networkx as nx_mod
print("RDKit:", rdkit.__version__)
print("NetworkX:", nx_mod.__version__)
print("SynKit available:", HAS_SYNKit)


In [ ]:
df = pd.read_csv("data/reactions_mapped.csv")
row = df.iloc[0]
react, prod = row.am_rxn_smiles.split(">>")

def draw_mapnums(smiles: str, size=(520,280)):
    m = Chem.MolFromSmiles(smiles)
    for a in m.GetAtoms():
        amap = a.GetAtomMapNum()
        if amap:
            a.SetProp("atomNote", str(amap))
    return Draw.MolToImage(m, size=size)

display(draw_mapnums(react))
display(draw_mapnums(prod))


In [ ]:
# Tiny CRN: nodes are SMILES strings, edges carry a label
G = nx.DiGraph()
G.add_node(react, kind="species")
G.add_node(prod, kind="species")
G.add_edge(react, prod, rule_id="toy_rule", source_rxn=row.rxn_id)

# Export DOT
try:
    from networkx.drawing.nx_pydot import write_dot
    dot_path = OUT / "S08_tiny_crn.dot"
    write_dot(G, dot_path.as_posix())
    print("Wrote", dot_path)
except Exception as e:
    print("DOT export skipped:", e)

# Quick schematic draw
try:
    import matplotlib.pyplot as plt
    pos = nx.spring_layout(G, seed=0)
    nx.draw(G, pos, with_labels=False, node_size=1200)
    edge_labels = nx.get_edge_attributes(G, "rule_id")
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=8)
    plt.title("Tiny CRN (toy)")
    plt.show()
except Exception as e:
    print("matplotlib draw skipped:", e)


# Discussion
- For larger networks, you will want canonical SMILES for node merging.
- Store provenance on edges: rule_id, source reaction, parameters (radius, filters).


# Quiz
1. What provenance fields would you add to CRN edges?
2. How would you merge identical species nodes in the CRN?
3. Suggest one way to visualize rule L/K/R as a three-panel figure.


# References and further reading

*Suggested citation style:*  
* Keyword: <i>Source</i> (year) (link)

- RDKit documentation: <i>RDKit</i> (ongoing) — https://www.rdkit.org/docs/
- RDKit Book: <i>The RDKit Book</i> (ongoing) — https://www.rdkit.org/docs/Book.html
- NetworkX documentation: <i>NetworkX</i> (ongoing) — https://networkx.org/documentation/stable/
- Graphviz DOT language: <i>Graphviz</i> (ongoing) — https://graphviz.org/documentation/
